# Validate the composed annotation SQLite store

This notebook validates a composed kidney dataset through the public `AnnotationReader.load(...)` API. It checks the merged classes, source references, pixel mapping, and SQLite tables. It does not download data or recreate the composition.

In [5]:
import os
from pathlib import Path

repository_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
os.chdir(repository_root)

In [6]:
os.getcwd()

'/home/max/repositories/MSIAutoEncoderWrapper'

In [7]:
from pathlib import Path
import json
import sqlite3

from msi_dataset_manager.annotations import AnnotationReader

workspace = Path('data/kidney_workspace')
cohort_id = 'kidney'
cohort_dir = workspace / 'datasets' / cohort_id
merged_imzml = cohort_dir / f'{cohort_id}.imzML'
merged_ibd = cohort_dir / f'{cohort_id}.ibd'
merged_sqlite = cohort_dir / f'{cohort_id}.sqlite'
selection_path = workspace / 'configs' / 'datasets' / cohort_id / 'selection.json'

for path in (merged_imzml, merged_ibd, merged_sqlite):
    assert path.is_file(), f'Missing composition output: {path}'

selection = json.loads(selection_path.read_text(encoding='utf-8'))
print(f'Selected source datasets: {len(selection["datasets"])}')
print(f'Merged SQLite: {merged_sqlite}')

Selected source datasets: 30
Merged SQLite: data/kidney_workspace/datasets/kidney/kidney.sqlite


## Load the merged reader

The wrapper uses this same reader object. `type='merge'` means that the reader reads the final SQLite rather than parsing provider files.

In [8]:
reader = AnnotationReader.load(
    type='merge',
    path=merged_sqlite,
    image_path=merged_imzml,
)
metadata = reader.get_dataset_metadata()
print(f'Source datasets in SQLite: {len(metadata["datasets"])}')
for dataset in metadata['datasets']:
    print(dataset['dataset_index'], dataset['dataset_id'], dataset['source'])

Source datasets in SQLite: 30
1 2025-07-06_22h07m04s metaspace
2 2025-07-06_16h31m32s metaspace
3 2017-04-20_09h06m22s metaspace
4 2024-04-10_17h06m19s metaspace
5 2024-05-23_14h25m01s metaspace
6 2024-02-20_01h41m56s metaspace
7 2024-02-20_01h38m59s metaspace
8 2024-02-20_01h45m20s metaspace
9 2024-02-20_01h44m07s metaspace
10 2024-02-19_05h42m55s metaspace
11 2024-02-20_01h43m24s metaspace
12 2024-02-20_01h40m11s metaspace
13 2024-02-20_01h46m58s metaspace
14 2024-02-19_05h43m55s metaspace
15 2024-02-20_01h46m10s metaspace
16 2024-02-20_01h54m41s metaspace
17 2024-02-19_05h21m14s metaspace
18 2019-03-19_17h21m15s metaspace
19 2024-02-20_01h49m35s metaspace
20 2024-02-19_05h02m33s metaspace
21 2024-02-20_01h48m46s metaspace
22 2024-02-19_05h36m42s metaspace
23 2024-02-20_01h57m32s metaspace
24 2024-02-19_04h23m32s metaspace
25 2024-02-20_01h55m56s metaspace
26 2026-04-22_21h03m00s metaspace
27 2024-02-20_01h53m31s metaspace
28 2024-02-20_01h42m42s metaspace
29 2024-02-19_05h12m47s met

## Inspect merged classes

`merged_annotations` contains `formula + adduct` and the merged pixel indices. It intentionally does not contain a global m/z.

In [9]:
merged_annotations = reader.get_annotations()
assert merged_annotations, 'No merged annotation classes were stored.'
print(f'Merged classes: {len(merged_annotations)}')
for annotation in merged_annotations[:10]:
    print(
        annotation['merged_annotation_id'],
        annotation['formula'],
        annotation['adduct'],
        len(annotation['spectrum_ids']),
    )
assert all('mz' not in annotation for annotation in merged_annotations)

Merged classes: 566
1 C10H12N2O5 -H 186
2 C10H12N4O5 -H 2735
3 C10H13N4O8P -H 78536
4 C10H14N4O5 -H 15797
5 C10H14N5O7P -H 83767
6 C10H14N5O8P -H 76095
7 C10H15N3O5 +Cl 15532
8 C10H15N5O10P2 -H 69669
9 C10H15NO2S -H 9963
10 C10H16N2O7 -H 7649


## Follow one merged pixel back to its source

The spectrum metadata gives the source dataset and source spectrum index. The annotations for that pixel then provide the source-specific m/z, FDR, database, and original METASPACE record.

In [10]:
merged_pixel = merged_annotations[0]['spectrum_ids'][0]
pixel_metadata = reader.get_spectrum_metadata(merged_pixel)
pixel_annotations = reader.get_spectrum_annotations(merged_pixel)

print('Merged pixel:', merged_pixel)
print('Source dataset:', pixel_metadata['dataset_id'])
print('Source spectrum:', pixel_metadata['source_spectrum_id'])
print('References:', len(pixel_annotations))
for reference in pixel_annotations:
    print({
        'formula': reference['formula'],
        'adduct': reference['adduct'],
        'mz': reference['mz'],
        'fdr': reference['fdr'],
        'database': reference['database_name'],
    })
assert pixel_annotations
assert all(reference['mz'] is not None for reference in pixel_annotations)

Merged pixel: 184097
Source dataset: 2024-05-23_14h25m01s
Source spectrum: 161
References: 132
{'formula': 'C10H12N2O5', 'adduct': '-H', 'mz': 239.067345049, 'fdr': 0.1, 'database': 'HMDB'}
{'formula': 'C10H15NO2S', 'adduct': '-H', 'mz': 212.075073455, 'fdr': 0.1, 'database': 'DrugBank'}
{'formula': 'C10H18O4', 'adduct': '-H', 'mz': 201.11323262099998, 'fdr': 0.1, 'database': 'HMDB'}
{'formula': 'C10H18O4', 'adduct': '-H', 'mz': 201.11323262099998, 'fdr': 0.1, 'database': 'DrugBank'}
{'formula': 'C10H21N3O', 'adduct': '[M]-', 'mz': 199.169010889, 'fdr': 0.05, 'database': 'DrugBank'}
{'formula': 'C10H7N3S', 'adduct': '[M]-', 'mz': 201.036616991, 'fdr': 0.1, 'database': 'DrugBank'}
{'formula': 'C11H20O4', 'adduct': '-H', 'mz': 215.12888268499998, 'fdr': 0.05, 'database': 'HMDB'}
{'formula': 'C11H20O4', 'adduct': '-H', 'mz': 215.12888268499998, 'fdr': 0.1, 'database': 'LipidMaps'}
{'formula': 'C12H24O2', 'adduct': '-H', 'mz': 199.170353573, 'fdr': 0.05, 'database': 'HMDB'}
{'formula': 'C1

## Validate the physical SQLite schema

This checks that the old catalog tables and the removed intensity BLOB are not present. Intensities remain in the imzML/ibd pair and are not copied into SQLite.

In [11]:
with sqlite3.connect(merged_sqlite) as connection:
    tables = {
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )
    }

print(sorted(tables))
assert {'datasets_metadata', 'pixel_segments', 'merged_annotations'} <= tables
assert 'datasets' not in tables
assert 'annotations' not in tables
assert 'spectrum_annotations' not in tables
assert 'annotation_materializations' not in tables
assert 'spectrum_mappings' not in tables
print('SQLite validation passed.')

['datasets_metadata', 'merged_annotations', 'pixel_segments', 'reference_annotations_0001', 'reference_annotations_0002', 'reference_annotations_0003', 'reference_annotations_0004', 'reference_annotations_0005', 'reference_annotations_0006', 'reference_annotations_0007', 'reference_annotations_0008', 'reference_annotations_0009', 'reference_annotations_0010', 'reference_annotations_0011', 'reference_annotations_0012', 'reference_annotations_0013', 'reference_annotations_0014', 'reference_annotations_0015', 'reference_annotations_0016', 'reference_annotations_0017', 'reference_annotations_0018', 'reference_annotations_0019', 'reference_annotations_0020', 'reference_annotations_0021', 'reference_annotations_0022', 'reference_annotations_0023', 'reference_annotations_0024', 'reference_annotations_0025', 'reference_annotations_0026', 'reference_annotations_0027', 'reference_annotations_0028', 'reference_annotations_0029', 'reference_annotations_0030']
SQLite validation passed.
